In [3]:
# ============================================================
# BM25 RETRIEVAL TEST
# ============================================================

import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "rag":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from rag.bm25_retriever import BM25Retriever

# Initialize BM25
bm25 = BM25Retriever()

# Test query
query = "Explain object detection in computer vision."

# Retrieve top 10
results = bm25.retrieve(
    query,
    top_k=10
)

print("\n" + "=" * 80)
print("BM25 TEST")
print("=" * 80)

print("\nQUERY:")
print(query)

print("\nRESULTS:")

for rank, result in enumerate(results, 1):

    print(
        f"{rank}. "
        f"[{result['score']:.4f}] "
        f"{result['id']} - "
        f"{result['metadata']['question']}"
    )

INITIALIZING BM25

Loading question data...
Records loaded: 2150
Questions extracted: 2150

Building BM25 index...
BM25 index built successfully.

BM25 READY

BM25 TEST

QUERY:
Explain object detection in computer vision.

RESULTS:
1. [7.9324] cs_fundamentals_networking_q007 - What is encapsulation in computer networking?
2. [7.2444] cybersecurity_cybersecurity_q052 - What is Endpoint Detection and Response (EDR)?
3. [7.0828] cs_fundamentals_oop_q047 - What is the difference between object identity and object equality?
4. [7.0340] ai_ml_computer_vision_q023 - What is a bounding box in object detection?
5. [6.8252] ai_ml_computer_vision_q021 - What is object detection?
6. [6.8252] cloud_cloud_basics_q023 - What is object storage?
7. [6.6902] ai_ml_computer_vision_q027 - What is mean Average Precision (mAP) in object detection?
8. [6.4535] ai_ml_computer_vision_q044 - How can data leakage occur in a computer vision dataset?
9. [6.2314] ai_ml_computer_vision_q042 - What is the difference 

In [6]:
from pathlib import Path

# ============================================================
# FIND PROJECT ROOT
# ============================================================

CURRENT_DIR = Path.cwd()

print("Current directory:")
print(CURRENT_DIR)

# If notebook is inside evaluation/ or evaluation/tests/
if CURRENT_DIR.name == "tests":
    PROJECT_ROOT = CURRENT_DIR.parent.parent
elif CURRENT_DIR.name == "evaluation":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

print("\nProject root:")
print(PROJECT_ROOT)

# ============================================================
# EVALUATION FILE
# ============================================================

EVAL_FILE = PROJECT_ROOT / "evaluation" / "tests" / "retrieval_eval.json"

print("\nEvaluation file:")
print(EVAL_FILE)

print("\nFile exists:", EVAL_FILE.exists())

Current directory:
C:\Users\User\RAG SYS\Question_Generation\evaluation

Project root:
C:\Users\User\RAG SYS\Question_Generation

Evaluation file:
C:\Users\User\RAG SYS\Question_Generation\evaluation\tests\retrieval_eval.json

File exists: False


In [7]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

print("Searching for retrieval_eval.json...")
print("=" * 70)

matches = list(PROJECT_ROOT.rglob("retrieval_eval.json"))

if matches:
    print(f"\nFound {len(matches)} file(s):\n")

    for path in matches:
        print(path)

else:
    print("\n❌ retrieval_eval.json was not found.")

Searching for retrieval_eval.json...

Found 1 file(s):

C:\Users\User\RAG SYS\Question_Generation\tests\retrieval_eval.json


In [9]:
import json
from pathlib import Path

EVAL_FILE = Path(
    r"C:\Users\User\RAG SYS\Question_Generation\tests\retrieval_eval.json"
)

with open(EVAL_FILE, "r", encoding="utf-8") as f:
    evaluation_data = json.load(f)

print("Number of records:", len(evaluation_data))
print("\nFirst record:")
print(evaluation_data[0])

print("\nKeys:")
print(evaluation_data[0].keys())

Number of records: 200

First record:
{'query': 'What is Computer Vision and what problems does it solve?', 'expected_id': 'ai_ml_computer_vision_q001', 'source_file': 'ai_ml\\computer_vision.md'}

Keys:
dict_keys(['query', 'expected_id', 'source_file'])


In [10]:
# ============================================================
# BM25 — PARAPHRASE RETRIEVAL EVALUATION
# ============================================================

import json
from pathlib import Path

# ============================================================
# LOAD EVALUATION DATA
# ============================================================

EVAL_FILE = Path(
    r"C:\Users\User\RAG SYS\Question_Generation\tests\retrieval_eval.json"
)

with open(EVAL_FILE, "r", encoding="utf-8") as f:
    evaluation_data = json.load(f)

print("=" * 70)
print("BM25 RETRIEVAL EVALUATION")
print("=" * 70)

print("\nEvaluation queries:", len(evaluation_data))


# ============================================================
# INITIALIZE BM25
# ============================================================

from rag.bm25_retriever import BM25Retriever

bm25 = BM25Retriever()


# ============================================================
# METRICS
# ============================================================

recall_at_1 = 0
recall_at_3 = 0
recall_at_5 = 0

reciprocal_ranks = []

total = len(evaluation_data)


# ============================================================
# RUN EVALUATION
# ============================================================

for i, item in enumerate(evaluation_data, 1):

    query = item["query"]
    expected_id = item["expected_id"]

    results = bm25.retrieve(
        query,
        top_k=5
    )

    retrieved_ids = [
        result["id"]
        for result in results
    ]

    # --------------------------------------------------------
    # Recall@1
    # --------------------------------------------------------

    if expected_id in retrieved_ids[:1]:
        recall_at_1 += 1

    # --------------------------------------------------------
    # Recall@3
    # --------------------------------------------------------

    if expected_id in retrieved_ids[:3]:
        recall_at_3 += 1

    # --------------------------------------------------------
    # Recall@5
    # --------------------------------------------------------

    if expected_id in retrieved_ids[:5]:
        recall_at_5 += 1

    # --------------------------------------------------------
    # MRR
    # --------------------------------------------------------

    if expected_id in retrieved_ids:

        rank = retrieved_ids.index(expected_id) + 1

        reciprocal_ranks.append(1 / rank)

    else:

        reciprocal_ranks.append(0)

    # --------------------------------------------------------
    # Progress
    # --------------------------------------------------------

    if i % 20 == 0 or i == total:

        print(
            f"Processed {i}/{total} "
            f"({i / total * 100:.1f}%)"
        )


# ============================================================
# CALCULATE METRICS
# ============================================================

recall_1 = recall_at_1 / total
recall_3 = recall_at_3 / total
recall_5 = recall_at_5 / total

mrr = sum(reciprocal_ranks) / total


# ============================================================
# FINAL RESULTS
# ============================================================

print("\n")
print("=" * 70)
print("BM25 FINAL RESULTS")
print("=" * 70)

print(f"\nTotal queries: {total}")

print(f"Recall@1: {recall_1 * 100:.2f}%")
print(f"Recall@3: {recall_3 * 100:.2f}%")
print(f"Recall@5: {recall_5 * 100:.2f}%")
print(f"MRR:      {mrr:.4f}")

print("\n" + "=" * 70)

BM25 RETRIEVAL EVALUATION

Evaluation queries: 200
INITIALIZING BM25

Loading question data...
Records loaded: 2150
Questions extracted: 2150

Building BM25 index...
BM25 index built successfully.

BM25 READY
Processed 20/200 (10.0%)
Processed 40/200 (20.0%)
Processed 60/200 (30.0%)
Processed 80/200 (40.0%)
Processed 100/200 (50.0%)
Processed 120/200 (60.0%)
Processed 140/200 (70.0%)
Processed 160/200 (80.0%)
Processed 180/200 (90.0%)
Processed 200/200 (100.0%)


BM25 FINAL RESULTS

Total queries: 200
Recall@1: 97.50%
Recall@3: 100.00%
Recall@5: 100.00%
MRR:      0.9875



In [11]:
print("BM25 object attributes:")
print(bm25.__dict__.keys())

BM25 object attributes:
dict_keys(['questions', 'metadata', 'bm25'])


In [12]:
# ============================================================
# BM25 — FULL 2,150 QUESTION EVALUATION
# ============================================================

print("=" * 70)
print("BM25 FULL RETRIEVAL EVALUATION")
print("=" * 70)

total = len(bm25.questions)

print(f"\nTotal questions: {total}")

# ------------------------------------------------------------
# Metrics
# ------------------------------------------------------------

recall_at_1 = 0
recall_at_3 = 0
recall_at_5 = 0

reciprocal_ranks = []

# ------------------------------------------------------------
# Evaluate every question against the full BM25 index
# ------------------------------------------------------------

for i, question in enumerate(bm25.questions, 1):

    # Get the correct ID from metadata
    expected_id = bm25.metadata[i - 1]["id"]

    # Retrieve using the question itself
    results = bm25.retrieve(
        question,
        top_k=5
    )

    retrieved_ids = [
        result["id"]
        for result in results
    ]

    # Recall@1
    if expected_id in retrieved_ids[:1]:
        recall_at_1 += 1

    # Recall@3
    if expected_id in retrieved_ids[:3]:
        recall_at_3 += 1

    # Recall@5
    if expected_id in retrieved_ids[:5]:
        recall_at_5 += 1

    # MRR
    if expected_id in retrieved_ids:
        rank = retrieved_ids.index(expected_id) + 1
        reciprocal_ranks.append(1 / rank)
    else:
        reciprocal_ranks.append(0)

    # Progress
    if i % 100 == 0 or i == total:
        print(
            f"Processed {i}/{total} "
            f"({i / total * 100:.1f}%)"
        )


# ============================================================
# FINAL METRICS
# ============================================================

recall_1 = recall_at_1 / total
recall_3 = recall_at_3 / total
recall_5 = recall_at_5 / total

mrr = sum(reciprocal_ranks) / total


# ============================================================
# RESULTS
# ============================================================

print("\n")
print("=" * 70)
print("BM25 FULL RESULTS")
print("=" * 70)

print(f"\nTotal questions: {total}")

print(f"Recall@1: {recall_1 * 100:.2f}%")
print(f"Recall@3: {recall_3 * 100:.2f}%")
print(f"Recall@5: {recall_5 * 100:.2f}%")
print(f"MRR:      {mrr:.4f}")

print("\n" + "=" * 70)

BM25 FULL RETRIEVAL EVALUATION

Total questions: 2150
Processed 100/2150 (4.7%)
Processed 200/2150 (9.3%)
Processed 300/2150 (14.0%)
Processed 400/2150 (18.6%)
Processed 500/2150 (23.3%)
Processed 600/2150 (27.9%)
Processed 700/2150 (32.6%)
Processed 800/2150 (37.2%)
Processed 900/2150 (41.9%)
Processed 1000/2150 (46.5%)
Processed 1100/2150 (51.2%)
Processed 1200/2150 (55.8%)
Processed 1300/2150 (60.5%)
Processed 1400/2150 (65.1%)
Processed 1500/2150 (69.8%)
Processed 1600/2150 (74.4%)
Processed 1700/2150 (79.1%)
Processed 1800/2150 (83.7%)
Processed 1900/2150 (88.4%)
Processed 2000/2150 (93.0%)
Processed 2100/2150 (97.7%)
Processed 2150/2150 (100.0%)


BM25 FULL RESULTS

Total questions: 2150
Recall@1: 98.37%
Recall@3: 100.00%
Recall@5: 100.00%
MRR:      0.9918

